In [3]:
# ============================================================
# CELL 1 — INSTALL DEPENDENCIES
# ============================================================

!pip -q install -U sarvamai google-genai


In [4]:
# ============================================================
# CELL 2 — IMPORTS + API KEYS + CLIENTS
# ============================================================

import base64

from google import genai
from google.genai import types

from sarvamai import SarvamAI

from google.colab import userdata
from google.colab import output

from IPython.display import Audio, display


# ------------------------------------------------------------
# API KEYS
# ------------------------------------------------------------

SARVAM_API_KEY = userdata.get("SARVAM_API_KEY").strip()
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY").strip()


# ------------------------------------------------------------
# SARVAM CLIENT
# ------------------------------------------------------------

sarvam_client = SarvamAI(
    api_subscription_key=SARVAM_API_KEY
)


# ------------------------------------------------------------
# GEMINI CLIENT
# ------------------------------------------------------------

gemini_client = genai.Client(
    api_key=GEMINI_API_KEY
)


# ------------------------------------------------------------
# MODEL
# ------------------------------------------------------------

GEMINI_MODEL = "gemini-3.5-flash-lite"


print("Sarvam client initialized ✅")
print("Gemini client initialized ✅")
print("Gemini model:", GEMINI_MODEL)

Sarvam client initialized ✅
Gemini client initialized ✅
Gemini model: gemini-3.5-flash-lite


In [5]:
# ============================================================
# CELL 3 — TEST GEMINI
# ============================================================

response = gemini_client.models.generate_content(
    model=GEMINI_MODEL,
    contents="Say hello as ArogyaVani in one short sentence."
)

print(response.text)

Hello, I am ArogyaVani, your trusted health companion ready to assist you on your wellness journey!


In [7]:
# ============================================================
# CELL 4 — AROGYAVANI SAFETY CONFIGURATION
# ============================================================

SUPPORTED_LANGUAGES = {
    "as-IN",
    "bn-IN",
    "brx-IN",
    "doi-IN",
    "gu-IN",
    "hi-IN",
    "kn-IN",
    "kok-IN",
    "ks-IN",
    "mai-IN",
    "ml-IN",
    "mni-IN",
    "mr-IN",
    "ne-IN",
    "od-IN",
    "pa-IN",
    "sa-IN",
    "sat-IN",
    "sd-IN",
    "ta-IN",
    "te-IN",
    "ur-IN",
    "en-IN"
}


MEDICAL_ADVICE_KEYWORDS = [
    "diagnose me",
    "diagnose this",
    "what disease do i have",
    "which disease do i have",
    "tell me my disease",
    "what medicine should i take",
    "which medicine should i take",
    "what tablet should i take",
    "which tablet should i take",
    "what dosage should i take",
    "what dose should i take",
    "how much medicine should i take",
    "prescribe medicine",
    "give me a prescription",
    "what treatment should i take",
    "which treatment should i take"
]


MEDICAL_SAFETY_RESPONSE = (
    "I'm sorry, but I cannot provide a diagnosis, prescribe medicines, "
    "or recommend medical treatments. I can help you with healthcare "
    "services, government health schemes, and healthcare facilities. "
    "For medical advice, please consult a qualified healthcare professional."
)

In [21]:
# ============================================================
# CELL 5 — GEMINI AROGYAVANI SYSTEM PROMPT
# ============================================================

AROGYAVANI_SYSTEM_PROMPT = """
You are ArogyaVani AI, a healthcare-access assistant for citizens of India.

You can help users with:

- Government healthcare schemes
- Scheme eligibility and required documents when trusted information
  is available
- Healthcare services and healthcare access
- Hospitals, PHCs and clinics
- General health information
- General information about common symptoms and health conditions
- Basic, low-risk self-care and comfort information
- General warning signs that may require medical attention
- Guidance about when to visit a PHC, clinic, hospital or qualified
  healthcare professional

IMPORTANT SAFETY LIMITATIONS:

You MUST NOT:

- Diagnose a disease or medical condition.
- Claim that the user definitely has a disease.
- Prescribe medicines.
- Recommend specific medicines.
- Give medicine dosages.
- Provide prescriptions.
- Recommend personalized medical treatments.
- Recommend potentially unsafe or unverified remedies.
- Invent government schemes.
- Invent scheme eligibility rules.
- Invent scheme benefits or required documents.
- Invent hospitals, PHCs, clinics, addresses, phone numbers,
  availability or facility information.
- Present uncertain information as confirmed fact.

YOU MAY provide:

- General, low-risk health information.
- Basic self-care and comfort measures for common minor symptoms,
  such as getting adequate rest, drinking enough fluids and staying
  comfortable.
- General information about common symptoms without diagnosing the user.
- General information about when professional medical care may be needed.

When discussing symptoms, do NOT state or imply that the user definitely
has a particular disease or condition.

Use careful language such as:
- "If this is a mild cold..."
- "These symptoms can sometimes occur with..."
- "For general comfort, you may..."

If the user asks for a diagnosis, prescription, specific medicine,
dosage, or personalized treatment plan, politely explain that ArogyaVani
cannot provide those services and recommend consulting a qualified
healthcare professional.

If the user describes severe, worsening, or concerning symptoms such as
difficulty breathing or chest pain, recommend seeking appropriate medical
attention promptly.

For government healthcare schemes, use only trusted information provided
by the application. Never invent scheme information.

For hospitals, PHCs or healthcare facilities, only provide information
available from trusted facility data. Never invent locations or availability.

Keep responses:
- Short
- Clear
- Simple
- Reassuring
- Natural
- Relevant to the user's question

Do not give unnecessary safety disclaimers when they are not relevant.

Never invent factual information.
"""

In [22]:
# ============================================================
# CELL 6 — GEMINI RESPONSE FUNCTION
# ============================================================

def ask_gemini(question):

    response = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=question,
        config=types.GenerateContentConfig(
            system_instruction=AROGYAVANI_SYSTEM_PROMPT,
            max_output_tokens=150
        )
    )

    if not response.text:
        raise RuntimeError(
            "Gemini returned an empty response."
        )

    return response.text.strip()


print("Gemini ArogyaVani function ready ✅")

Gemini ArogyaVani function ready ✅


In [23]:
# ============================================================
# CELL 7 — TRANSLATION FUNCTIONS
# ============================================================

def translate_to_english(text, language_code):

    if language_code == "en-IN":
        return text

    response = sarvam_client.text.translate(
        input=text,
        source_language_code=language_code,
        target_language_code="en-IN"
    )

    return response.translated_text.strip()


def translate_from_english(text, target_language):

    if target_language == "en-IN":
        return text

    response = sarvam_client.text.translate(
        input=text,
        source_language_code="en-IN",
        target_language_code=target_language
    )

    return response.translated_text.strip()


print("Translation functions ready ✅")

Translation functions ready ✅


In [43]:
# ============================================================
# AROGYAVANI — CONTROLLED HEALTHCARE ACCESS ROUTER
# ============================================================

HEALTHCARE_KEYWORDS = [
    "health",
    "healthcare",
    "hospital",
    "clinic",
    "doctor",
    "medical",
    "medicine",
    "symptom",
    "cold",
    "fever",
    "cough",
    "scheme",
    "ayushman",
    "pmjay",
    "phc",
    "health facility",
    "healthcare facility",
    "government health",
    "healthcare scheme",
    "eligibility",
    "health card",
    "ration card",
    "health insurance"
]


LOCATION_KEYWORDS = [
    "near me",
    "nearby",
    "nearest",
    "closest",
    "around me",
    "near my location",
    "near my area",
    "hospital near",
    "hospital nearby",
    "nearest hospital",
    "closest hospital",
    "phc near",
    "phc nearby",
    "nearest phc",
    "clinic near",
    "clinic nearby",
    "nearest clinic",
    "healthcare near",
    "healthcare nearby",
    "health facility near",
    "health facility nearby"
]


MEDICAL_ADVICE_KEYWORDS = [
    "diagnose me",
    "diagnose this",
    "what disease do i have",
    "which disease do i have",
    "tell me my disease",
    "what medicine should i take",
    "which medicine should i take",
    "what tablet should i take",
    "which tablet should i take",
    "what dosage should i take",
    "what dose should i take",
    "how much medicine should i take",
    "prescribe medicine",
    "give me a prescription",
    "what treatment should i take",
    "which treatment should i take",
    "what should i take for",
    "what medicine can i take"
]


# ============================================================
# FIXED RESPONSES
# ============================================================

MEDICAL_SAFETY_RESPONSE = (
    "I'm sorry, but I cannot provide a diagnosis, prescribe medicines, "
    "or recommend medical treatments. I can help you with healthcare "
    "services, government health schemes, and healthcare facilities. "
    "For medical advice, please consult a qualified healthcare professional."
)


LOCATION_RESPONSE = (
    "Sure, I can help you find a nearby healthcare facility. "
    "Please provide your location or allow location access so I can "
    "help identify the nearest suitable hospital, PHC, or healthcare facility."
)


OUT_OF_SCOPE_RESPONSE = (
    "I'm sorry, I can only assist with healthcare access, government "
    "health schemes, and healthcare facilities. Please ask me something "
    "related to healthcare services or schemes."
)


# ============================================================
# KEYWORD CHECK
# ============================================================

def contains_keyword(text, keywords):

    text = text.lower().strip()

    return any(
        keyword in text
        for keyword in keywords
    )


# ============================================================
# INTENT DETECTION
# ============================================================

def detect_intent(english_text):

    # Medical safety gets highest priority
    if contains_keyword(
        english_text,
        MEDICAL_ADVICE_KEYWORDS
    ):
        return "medical_advice"

    # Location / nearby facility request
    if contains_keyword(
        english_text,
        LOCATION_KEYWORDS
    ):
        return "location"

    # General healthcare request
    if contains_keyword(
        english_text,
        HEALTHCARE_KEYWORDS
    ):
        return "healthcare"

    # Anything unrelated
    return "out_of_scope"


# ============================================================
# GEMINI
# ============================================================

def ask_gemini(question):

    response = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=question,
        config=types.GenerateContentConfig(
            system_instruction=AROGYAVANI_SYSTEM_PROMPT,
            max_output_tokens=150
        )
    )

    if not response.text:
        raise RuntimeError(
            "Gemini returned an empty response."
        )

    return response.text.strip()


# ============================================================
# MAIN AROGYAVANI ROUTER
# ============================================================

def arogyavani_router(user_text, detected_language):

    # --------------------------------------------------------
    # STEP 1 — Translate to English for routing
    # --------------------------------------------------------

    english_text = translate_to_english(
        user_text,
        detected_language
    )

    print("\n🌐 English routing text:")
    print(english_text)


    # --------------------------------------------------------
    # STEP 2 — Detect intent
    # --------------------------------------------------------

    intent = detect_intent(
        english_text
    )

    print("\n🧭 Detected intent:")
    print(intent)


    # --------------------------------------------------------
    # STEP 3 — Medical advice
    # --------------------------------------------------------

    if intent == "medical_advice":

        return MEDICAL_SAFETY_RESPONSE


    # --------------------------------------------------------
    # STEP 4 — Location request
    # --------------------------------------------------------

    if intent == "location":

        return LOCATION_RESPONSE


    # --------------------------------------------------------
    # STEP 5 — Healthcare question → Gemini
    # --------------------------------------------------------

    if intent == "healthcare":

        return ask_gemini(
            english_text
        )


    # --------------------------------------------------------
    # STEP 6 — Unrelated question
    # --------------------------------------------------------

    return OUT_OF_SCOPE_RESPONSE

In [47]:
# ============================================================
# CELL 9 — VOICE RECORDING
# ============================================================

print("🎤 Recording for 5 seconds...")
print("Please speak now.")

audio_base64 = output.eval_js("""
(async () => {

    const stream = await navigator.mediaDevices.getUserMedia({
        audio: true
    });

    try {

        const recorder = new MediaRecorder(stream);
        const chunks = [];

        recorder.ondataavailable = (event) => {

            if (event.data && event.data.size > 0) {
                chunks.push(event.data);
            }

        };

        recorder.start();

        await new Promise(resolve => {
            setTimeout(resolve, 5000);
        });

        await new Promise(resolve => {

            recorder.onstop = resolve;
            recorder.stop();

        });

        const blob = new Blob(chunks, {
            type: "audio/webm"
        });

        const buffer = await blob.arrayBuffer();
        const bytes = new Uint8Array(buffer);

        let binary = "";
        const chunkSize = 0x8000;

        for (
            let i = 0;
            i < bytes.length;
            i += chunkSize
        ) {

            const chunk = bytes.subarray(
                i,
                Math.min(i + chunkSize, bytes.length)
            );

            binary += String.fromCharCode(...chunk);

        }

        return btoa(binary);

    } finally {

        stream.getTracks().forEach(track => {
            track.stop();
        });

    }

})()
""")


if not audio_base64:
    raise RuntimeError(
        "No audio was recorded."
    )


audio_bytes = base64.b64decode(
    audio_base64
)


input_audio_path = (
    "/content/arogyavani_input.webm"
)


with open(input_audio_path, "wb") as audio_file:

    audio_file.write(audio_bytes)


print("✅ Voice recording completed.")

🎤 Recording for 5 seconds...
Please speak now.
✅ Voice recording completed.


In [48]:
# ============================================================
# CELL 10 — SARVAM SAARAS STT
# ============================================================

print("📝 Converting speech to text...")

with open(input_audio_path, "rb") as audio_file:

    stt_response = sarvam_client.speech_to_text.transcribe(
        file=audio_file,
        model="saaras:v3",
        mode="transcribe"
    )


user_text = (
    stt_response.transcript or ""
).strip()


detected_language = (
    stt_response.language_code
)


if not user_text:

    raise RuntimeError(
        "Saaras returned empty transcription."
    )


print("\n📝 You said:")
print(user_text)

print("\n🌐 Detected language:")
print(detected_language)

📝 Converting speech to text...

📝 You said:
Hello, suggest me nearby hospital location quick and what should I do to make my fever comfortable?

🌐 Detected language:
en-IN


In [49]:
# ============================================================
# CELL 11 — AROGYAVANI RESPONSE
# ============================================================

print("\n🤖 ArogyaVani is thinking...")


response_text = arogyavani_router(
    user_text,
    detected_language
)


print("\n💬 ArogyaVani:")
print(response_text)


# ------------------------------------------------------------
# Translate response back to user's language
# ------------------------------------------------------------

final_response = translate_from_english(
    response_text,
    detected_language
)


print("\n🌐 Final multilingual response:")
print(final_response)


🤖 ArogyaVani is thinking...

🌐 English routing text:
Hello, suggest me nearby hospital location quick and what should I do to make my fever comfortable?

🧭 Detected intent:
location

💬 ArogyaVani:
Sure, I can help you find a nearby healthcare facility. Please provide your location or allow location access so I can help identify the nearest suitable hospital, PHC, or healthcare facility.

🌐 Final multilingual response:
Sure, I can help you find a nearby healthcare facility. Please provide your location or allow location access so I can help identify the nearest suitable hospital, PHC, or healthcare facility.


In [50]:
# ============================================================
# CELL 12 — SARVAM BULBUL TTS
# ============================================================

print("\n🔊 Generating voice response...")


tts_response = sarvam_client.text_to_speech.convert(
    text=final_response,
    language_code=detected_language,
    model="bulbul:v3",
    speaker="priya"
)


if not tts_response.audios:

    raise RuntimeError(
        "Bulbul returned no audio."
    )


audio_base64 = tts_response.audios[0]

audio_bytes = base64.b64decode(
    audio_base64
)


output_audio_path = (
    "/content/arogyavani_final_response.wav"
)


with open(output_audio_path, "wb") as audio_file:

    audio_file.write(audio_bytes)


print("✅ Voice response generated.")


display(
    Audio(
        output_audio_path,
        autoplay=True
    )
)


🔊 Generating voice response...
✅ Voice response generated.


In [42]:
# ============================================================
# AROGYAVANI — COMPLETE MULTILINGUAL VOICE AGENT
# Sarvam Saaras → Translation → Safety Router → Gemini
# → Translation → Sarvam Bulbul
# ============================================================

import base64

from google.colab import output
from IPython.display import Audio, display


# ============================================================
# 1. AROGYAVANI CONFIGURATION
# ============================================================

SUPPORTED_LANGUAGES = {
    "as-IN", "bn-IN", "brx-IN", "doi-IN", "gu-IN",
    "hi-IN", "kn-IN", "kok-IN", "ks-IN", "mai-IN",
    "ml-IN", "mni-IN", "mr-IN", "ne-IN", "od-IN",
    "pa-IN", "sa-IN", "sat-IN", "sd-IN", "ta-IN",
    "te-IN", "ur-IN", "en-IN"
}


MEDICAL_ADVICE_KEYWORDS = [
    "diagnose me",
    "diagnose this",
    "what disease do i have",
    "which disease do i have",
    "tell me my disease",
    "what medicine should i take",
    "which medicine should i take",
    "what tablet should i take",
    "which tablet should i take",
    "what dosage should i take",
    "what dose should i take",
    "how much medicine should i take",
    "prescribe medicine",
    "give me a prescription",
    "what treatment should i take",
    "which treatment should i take"
]


MEDICAL_SAFETY_RESPONSE = (
    "I'm sorry, but I cannot provide a diagnosis, prescribe medicines, "
    "or recommend medical treatments. I can help you with healthcare "
    "services, government health schemes, and healthcare facilities. "
    "For medical advice, please consult a qualified healthcare professional."
)


AROGYAVANI_SYSTEM_PROMPT = """
You are ArogyaVani AI, a healthcare-access assistant for citizens of India.

You can help users with:

- Government healthcare schemes
- Scheme eligibility and required documents when trusted information
  is available
- Healthcare services and healthcare access
- Hospitals, PHCs and clinics
- General health information
- General information about common symptoms and health conditions
- Basic, low-risk self-care and comfort information
- General warning signs that may require medical attention
- Guidance about when to visit a PHC, clinic, hospital or qualified
  healthcare professional

IMPORTANT SAFETY LIMITATIONS:

You MUST NOT:

- Diagnose a disease or medical condition.
- Claim that the user definitely has a disease.
- Prescribe medicines.
- Recommend specific medicines.
- Give medicine dosages.
- Provide prescriptions.
- Recommend personalized medical treatments.
- Recommend potentially unsafe or unverified remedies.
- Invent government schemes.
- Invent scheme eligibility rules.
- Invent scheme benefits or required documents.
- Invent hospitals, PHCs, clinics, addresses, phone numbers,
  availability or facility information.
- Present uncertain information as confirmed fact.

YOU MAY provide:

- General, low-risk health information.
- Basic self-care and comfort measures for common minor symptoms,
  such as getting adequate rest, drinking enough fluids and staying
  comfortable.
- General information about common symptoms without diagnosing the user.
- General information about when professional medical care may be needed.

When discussing symptoms, do NOT state or imply that the user definitely
has a particular disease or condition.

Use careful language such as:
- "If this is a mild cold..."
- "These symptoms can sometimes occur with..."
- "For general comfort, you may..."

If the user asks for a diagnosis, prescription, specific medicine,
dosage, or personalized treatment plan, politely explain that ArogyaVani
cannot provide those services and recommend consulting a qualified
healthcare professional.

If the user describes severe, worsening, or concerning symptoms such as
difficulty breathing or chest pain, recommend seeking appropriate medical
attention promptly.

For government healthcare schemes, use only trusted information provided
by the application. Never invent scheme information.

For hospitals, PHCs or healthcare facilities, only provide information
available from trusted facility data. Never invent locations or availability.

Keep responses:
- Short
- Clear
- Simple
- Reassuring
- Natural
- Relevant to the user's question

Do not give unnecessary safety disclaimers when they are not relevant.

Never invent factual information.
"""


# ============================================================
# 2. TRANSLATION FUNCTIONS
# ============================================================

def translate_to_english(text, language_code):

    if language_code == "en-IN":
        return text

    response = sarvam_client.text.translate(
        input=text,
        source_language_code=language_code,
        target_language_code="en-IN"
    )

    return response.translated_text.strip()


def translate_from_english(text, target_language):

    if target_language == "en-IN":
        return text

    response = sarvam_client.text.translate(
        input=text,
        source_language_code="en-IN",
        target_language_code=target_language
    )

    return response.translated_text.strip()


# ============================================================
# 3. MEDICAL SAFETY ROUTER
# ============================================================

def is_medical_advice_question(text):

    text = text.lower().strip()

    return any(
        keyword in text
        for keyword in MEDICAL_ADVICE_KEYWORDS
    )


# ============================================================
# 4. GEMINI RESPONSE
# ============================================================

def ask_gemini(question):

    response = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=question,
        config=types.GenerateContentConfig(
            system_instruction=AROGYAVANI_SYSTEM_PROMPT,
            max_output_tokens=150
        )
    )

    if not response.text:
        raise RuntimeError(
            "Gemini returned an empty response."
        )

    return response.text.strip()


# ============================================================
# 5. RECORD VOICE
# ============================================================

print("🎤 ArogyaVani is ready!")
print("Speak after the microphone starts.")
print("Recording for 5 seconds...")

audio_base64 = output.eval_js("""
(async () => {

    const stream = await navigator.mediaDevices.getUserMedia({
        audio: true
    });

    try {

        const recorder = new MediaRecorder(stream);
        const chunks = [];

        recorder.ondataavailable = (event) => {

            if (event.data && event.data.size > 0) {
                chunks.push(event.data);
            }

        };

        recorder.start();

        await new Promise(resolve => {
            setTimeout(resolve, 5000);
        });

        await new Promise(resolve => {

            recorder.onstop = resolve;
            recorder.stop();

        });

        const blob = new Blob(chunks, {
            type: "audio/webm"
        });

        const buffer = await blob.arrayBuffer();
        const bytes = new Uint8Array(buffer);

        let binary = "";
        const chunkSize = 0x8000;

        for (
            let i = 0;
            i < bytes.length;
            i += chunkSize
        ) {

            const chunk = bytes.subarray(
                i,
                Math.min(i + chunkSize, bytes.length)
            );

            binary += String.fromCharCode(...chunk);

        }

        return btoa(binary);

    } finally {

        stream.getTracks().forEach(track => {
            track.stop();
        });

    }

})()
""")


if not audio_base64:
    raise RuntimeError("No audio was recorded.")


audio_bytes = base64.b64decode(audio_base64)

input_audio_path = "/content/arogyavani_input.webm"

with open(input_audio_path, "wb") as f:
    f.write(audio_bytes)

print("✅ Voice captured")


# ============================================================
# 6. SARVAM SAARAS — SPEECH TO TEXT
# ============================================================

print("\n📝 Converting speech to text...")

with open(input_audio_path, "rb") as audio_file:

    stt_response = sarvam_client.speech_to_text.transcribe(
        file=audio_file,
        model="saaras:v3",
        mode="transcribe"
    )


user_text = (
    stt_response.transcript or ""
).strip()

detected_language = stt_response.language_code


if not user_text:
    raise RuntimeError(
        "Saaras returned an empty transcript."
    )


print("\n📝 You said:")
print(user_text)

print("\n🌐 Detected language:")
print(detected_language)


# ============================================================
# 7. LANGUAGE VALIDATION
# ============================================================

if detected_language not in SUPPORTED_LANGUAGES:

    print(
        "\n⚠️ Detected language is not currently configured:"
    )
    print(detected_language)

    raise RuntimeError(
        f"Unsupported language: {detected_language}"
    )


# ============================================================
# 8. TRANSLATE USER TEXT → ENGLISH
# ============================================================

print("\n🌐 Translating for ArogyaVani...")

english_text = translate_to_english(
    user_text,
    detected_language
)

print("\n🔤 English text:")
print(english_text)


# ============================================================
# 9. SAFETY ROUTER → GEMINI
# ============================================================

print("\n🤖 ArogyaVani is thinking...")

if is_medical_advice_question(english_text):

    response_text = MEDICAL_SAFETY_RESPONSE

    print("\n🛡️ Safety router activated.")

else:

    response_text = ask_gemini(
        english_text
    )


print("\n💬 ArogyaVani response:")
print(response_text)


# ============================================================
# 10. TRANSLATE RESPONSE → USER'S LANGUAGE
# ============================================================

print("\n🌐 Preparing response in user's language...")

final_response = translate_from_english(
    response_text,
    detected_language
)

print("\n💬 Final response:")
print(final_response)


# ============================================================
# 11. SARVAM BULBUL — TEXT TO SPEECH
# ============================================================

print("\n🔊 Generating voice response...")

tts_response = sarvam_client.text_to_speech.convert(
    text=final_response,
    language_code=detected_language,
    model="bulbul:v3",
    speaker="priya"
)


if not tts_response.audios:
    raise RuntimeError(
        "Bulbul returned no audio."
    )


audio_base64 = tts_response.audios[0]

audio_bytes = base64.b64decode(
    audio_base64
)

output_audio_path = (
    "/content/arogyavani_final_response.wav"
)

with open(output_audio_path, "wb") as f:
    f.write(audio_bytes)


# ============================================================
# 12. PLAY RESPONSE
# ============================================================

print("\n🔊 ArogyaVani says:")
print(final_response)

display(
    Audio(
        output_audio_path,
        autoplay=True
    )
)

print("\n✅ Complete ArogyaVani voice pipeline finished!")

🎤 ArogyaVani is ready!
Speak after the microphone starts.
Recording for 5 seconds...
✅ Voice captured

📝 Converting speech to text...

📝 You said:
I have pneumonia, what should I do to make me feel comfortable? Or prescribe me medicine.

🌐 Detected language:
en-IN

🌐 Translating for ArogyaVani...

🔤 English text:
I have pneumonia, what should I do to make me feel comfortable? Or prescribe me medicine.

🤖 ArogyaVani is thinking...

💬 ArogyaVani response:
I cannot prescribe medicines, provide prescriptions, or recommend specific medical treatments. 

If you suspect you have pneumonia, it is very important to consult a qualified healthcare professional or visit a nearby clinic or hospital for a proper examination and diagnosis. 

For general comfort while you recover under medical guidance, you may:
- **Get plenty of rest** to help your body recover.
- **Stay hydrated** by drinking plenty of water and warm fluids.
- **Use a humidifier** or breathe in steam from a bowl of hot water to help


✅ Complete ArogyaVani voice pipeline finished!
